# Rawboost Pipeline

In [86]:
import numpy as np
from scipy import signal
from scipy.io import wavfile
import copy
import librosa
from IPython.display import Audio, display

In [87]:
def randRange(x1, x2, integer=False):
    """
    Generate a random number uniformly between x1 and x2.
    If integer=True, return the integer part.
    """
    y = np.random.uniform(low=x1, high=x2)
    return int(y) if integer else y


def normWav(x, always=False):
    """
    Normalize waveform so its max absolute value is <=1.
    If always=True, always normalize; otherwise only if >1.
    """
    if always or np.max(np.abs(x)) > 1:
        return x / np.max(np.abs(x))
    return x


def genNotchCoeffs(
    nBands, minF, maxF, minBW, maxBW,
    minCoeff, maxCoeff, minG, maxG, fs
):
    """
    Generate combined FIR notch filter coefficients across nBands bands,
    then scale by gain G (in dB) and normalize by max frequency response.
    """
    b = 1
    for _ in range(nBands):
        fc = randRange(minF, maxF)
        bw = randRange(minBW, maxBW)
        c = randRange(minCoeff, maxCoeff, integer=True)
        # ensure odd number of taps
        if c % 2 == 0:
            c += 1
        f1 = fc - bw / 2
        f2 = fc + bw / 2
        # clamp to valid range
        if f1 <= 0:
            f1 = 1e-3
        if f2 >= fs / 2:
            f2 = fs / 2 - 1e-3
        # design bandstop FIR
        band = signal.firwin(c, [float(f1), float(f2)], window='hamming', fs=fs)
        b = np.convolve(band, b)

    # random gain in dB
    G = randRange(minG, maxG, 0);
    # compute frequency response
    _, h = signal.freqz(b, 1, fs=fs)
    # scale filter coefficients by gain and normalize
    b = pow(10, G/20) * b / np.amax(abs(h))
    return b


def filterFIR(x, b):
    """
    Zero-pad, apply FIR filter b to signal x, and remove padding to maintain length.
    """
    N = b.shape[0] + 1
    xpad = np.pad(x, (0, N), 'constant')
    y = signal.lfilter(b, 1, xpad)
    # remove padding
    return y[int(N/2): int(y.shape[0] - N/2)]

In [88]:
def SSI_additive_noise(
    x, fs, SNRmin, SNRmax,
    nBands, minF, maxF, minBW, maxBW,
    minCoeff, maxCoeff, minG, maxG
):
    """
    Add stationary, signal-independent noise shaped by notch filters.
    """
    # white Gaussian noise
    noise = np.random.normal(0, 1, x.shape[0])
    # shape noise
    b = genNotchCoeffs(
        nBands, minF, maxF, minBW, maxBW,
        minCoeff, maxCoeff, minG, maxG, fs
    )
    noise = filterFIR(noise, b)
    noise = normWav(noise, 1)

    # random SNR
    SNR = randRange(SNRmin, SNRmax, 0)
    # scale noise to desired SNR
    noise = noise / np.linalg.norm(noise, 2) * np.linalg.norm(x, 2) / 10**(0.05 * SNR)
    # add to signal
    return x + noise

In [ ]:
def LnL_convolutive_noise(
    x, fs, N_f,
    nBands, minF, maxF, minBW, maxBW,
    minCoeff, maxCoeff, minG, maxG,
    minBiasLinNonLin, maxBiasLinNonLin
):
    """
    Add linear and non-linear convolutive noise by filtering x^k for k=1..N_f.
    """
    y = [0] * x.shape[0]
    for i in range(0, N_f):
        # adjust bias for the second filter
        if i == 1:
            minG = minG - minBiasLinNonLin
            maxG = maxG - maxBiasLinNonLin
        else:
            minG = minG
            maxG = maxG

        b = genNotchCoeffs(
            nBands, minF, maxF, minBW, maxBW,
            minCoeff, maxCoeff, minG, maxG, fs
        )
        y = y + filterFIR(np.power(x, (i+1)), b)

    # zero-mean
    y = y - np.mean(y)
    # normalize
    y = normWav(y, 0)
    return y

In [90]:
def ISD_additive_noise(x, P, g_sd):
    """
    Add impulsive, signal-dependent noise to random samples of x.
    """
    y = copy.deepcopy(x)
    beta = randRange(0, P, 0)
    x_len = x.shape[0]
    n = int(x_len * (beta / 100))
    p = np.random.permutation(x_len)[:n]
    f_r = np.multiply(((2*np.random.rand(p.shape[0]))-1), ((2*np.random.rand(p.shape[0]))-1))
    r = g_sd * x[p] * f_r
    y[p] = x[p] + r
    return normWav(y, 0)

### Load the audio and adjust parameters

In [99]:
# audio_path = "/home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_alaw.wav"
# audio_path = "/mnt/media/fair/audio/replay_attacks/datasets/ASVSpoof2017/ASVspoof2017_V2_train/ASVspoof2017_V2_train/T_1000001.wav"
audio_path = "/mnt/media/fair/audio/replay_attacks/datasets/ASVSpoof2019/PA/ASVspoof2019_PA_train/flac/PA_T_0000007.flac"
x, fs = librosa.load(audio_path, sr=16000, mono=True)

In [100]:
minG, maxG = 0, 0
N_f = 5
minBiasLinNonLin, maxBiasLinNonLin = 5, 20
P, g_sd = 10, 2
seg_samples = 16000
top_db = 30.0
SNRmin, SNRmax = 10, 40
fs = 16000
nBands, minF, maxF = 5, 20, 8000
minBW, maxBW = 100, 1000
minCoeff, maxCoeff = 10, 100

### Apply Rawboost

In [101]:
x_ssi = SSI_additive_noise(
    x, fs, SNRmin, SNRmax,
    nBands, minF, maxF, minBW, maxBW,
    minCoeff, maxCoeff, minG, maxG
)
x_conv = LnL_convolutive_noise(
    x, fs, N_f,
    nBands, minF, maxF, minBW, maxBW,
    minCoeff, maxCoeff, minG, maxG,
    minBiasLinNonLin, maxBiasLinNonLin
)
x_isd = ISD_additive_noise(x, P, g_sd)

In [102]:
print("Original:")
display(Audio(x, rate=fs))
print("SSI Noise:")
display(Audio(x_ssi, rate=fs))
print("Convolutive Noise:")
display(Audio(x_conv, rate=fs))
print("Impulsive Noise:")
display(Audio(x_isd, rate=fs))

Original:


SSI Noise:


Convolutive Noise:


Impulsive Noise:


# Vicomtech

In [ ]:
# Stationary signal independent noise
def SSI_additive_noise(self, x):
    # White noise
    noise = np.random.normal(0, 1, x.shape[0])
    # Coloured Noise (Notch Filter)
    b = self.genNotchCoeffs(self.minG, self.maxG)
    noise = self.filterFIR(noise, b)
    noise = self.normWav(noise, 1)
    # Gain parameter with random SNR
    SNR = self.randRange(self.SNRmin, self.SNRmax, 0)
    noise_norm = np.linalg.norm(noise, 2)
    x_norm = np.linalg.norm(x, 2)
    gain = noise_norm * x_norm / 10.0 ** (0.05 * SNR)
    # Stationary signal independent noise
    noise = noise / gain
    x = x + noise
    return x

In [ ]:
# Linear and non-Linear convolutive noise
def LnL_convolutive_noise(self, x):
    y = [0] * x.shape[0]
    min_G = self.minG
    max_G = self.maxG
    # Add different FIR filters with different orders
    for i in range(0, self.N_f):
        if i == 1:
            min_G = self.minG - self.minBiasLinNonLin
            max_G = self.maxG - self.maxBiasLinNonLin
        b = self.genNotchCoeffs(min_G, max_G)
        y = y + self.filterFIR(np.power(x, (i + 1)), b)
    y = y - np.mean(y)
    y = self.normWav(y, 0)
    return y

In [ ]:
# Impulsive signal dependent noise
def ISD_additive_noise(self, x):
    y = copy.deepcopy(x)
    # Randomly select a percentage of samples of the signal
    beta = self.randRange(0, self.P, 0)
    x_len = x.shape[0]
    n = int(x_len * (beta / 100))
    # Generate random indexes
    p = np.random.permutation(x_len)[:n]
    # Generate noise values
    v_1 = ((2 * np.random.rand(p.shape[0])) - 1)
    v_1 = ((2 * np.random.rand(p.shape[0])) - 1)
    f_r = self.g_sd * x[p] * f_r
    # Add impulsive noise to the selected samples
    y[p] = x[p] + r
    y = self.normWav(y, 0)
    return y

In [ ]:
class RawBoostAugment:
    def __init__(
        self, minG=0, maxG=0, N_f=5, minBiasLinNonLin=5, maxBiasLinNonLin=20,
        P=10, g_sd=2, seg_samples=16000, top_db=30.0, SNRmin=10, SNRmax=40,fs=16000,
        nBands=5,minF=20,maxF=8000,minBW=100,maxBW=1000,minCoeff=10,maxCoeff=100
    ):
        self.minG = minG
        self.maxG = maxG
        self.N_f = N_f
        self.minBiasLinNonLin = minBiasLinNonLin
        self.maxBiasLinNonLin = maxBiasLinNonLin
        self.P = P
        self.g_sd = g_sd
        self.seg_samples = seg_samples
        self.top_db = top_db
        self.SNRmin = SNRmin
        self.SNRmax = SNRmax
        self.fs = fs
        self.nBands = nBands
        self.minF = minF
        self.maxF = maxF
        self.minBW = minBW
        self.maxBW = maxBW
        self.minCoeff = minCoeff
        self.maxCoeff = maxCoeff

    # --- Funciones auxiliares ---
    def genNotchCoeffs(self, minG,maxG):
        b = 1
        for i in range(0, self.nBands):
            fc = self.randRange(self.minF,self.maxF,0)
            bw = self.randRange(self.minBW,self.maxBW,0)
            c = self.randRange(self.minCoeff,self.maxCoeff,1)
            
            if c/2 == int(c/2):
                c = c + 1
            f1 = fc - bw/2
            f2 = fc + bw/2
            if f1 <= 0:
                f1 = 1/1000
            if f2 >= self.fs/2:
                f2 =  self.fs/2-1/1000
            b = np.convolve(signal.firwin(c, [float(f1), float(f2)], window='hamming', fs=self.fs),b)

        G = self.randRange(minG,maxG,0);
        _, h = signal.freqz(b, 1, fs=self.fs)    
        b = pow(10, G/20)*b/np.amax(abs(h))   
        return b

    def filterFIR(self,x,b):
        N = b.shape[0] + 1
        xpad = np.pad(x, (0, N), 'constant')
        y = signal.lfilter(b, 1, xpad)
        y = y[int(N/2):int(y.shape[0]-N/2)]
        return y

    def normWav(self,x,always):
        if always:
            x = x/np.amax(abs(x))
        elif np.amax(abs(x)) > 1:
                x = x/np.amax(abs(x))
        return x

    def randRange(self,x1, x2, integer):
        y = np.random.uniform(low=x1, high=x2)
        if integer:
            y = int(y)
        return y

    # Linear and non-linear convolutive noise
    def LnL_convolutive_noise(self,x):
        y = [0] * x.shape[0]
        for i in range(0, self.N_f):
            if i == 1:
                minG = self.minG-self.minBiasLinNonLin
                maxG = self.maxG-self.maxBiasLinNonLin
            else:
                minG = self.minG
                maxG = self.maxG
            b = self.genNotchCoeffs(minG,maxG)
            y = y + self.filterFIR(np.power(x, (i+1)),  b)     
        y = y - np.mean(y)
        y = self.normWav(y,0)
        return y

    # Impulsive signal dependent noise
    def ISD_additive_noise(self,x):
        beta = self.randRange(0, self.P, 0)
        
        y = copy.deepcopy(x)
        x_len = x.shape[0]
        n = int(x_len*(beta/100))
        p = np.random.permutation(x_len)[:n]
        f_r= np.multiply(((2*np.random.rand(p.shape[0]))-1),((2*np.random.rand(p.shape[0]))-1))
        r = self.g_sd * x[p] * f_r
        y[p] = x[p] + r
        y = self.normWav(y,0)
        return y

    # Stationary signal independent noise
    def SSI_additive_noise(self,x):
        noise = np.random.normal(0, 1, x.shape[0])
        b = self.genNotchCoeffs(self.minG,self.maxG)
        noise = self.filterFIR(noise, b)
        noise = self.normWav(noise,1)
        SNR = self.randRange(self.SNRmin, self.SNRmax, 0)
        noise = noise / np.linalg.norm(noise,2) * np.linalg.norm(x,2) / 10.0**(0.05 * SNR)
        x = x + noise
        return x
    
    # --- Segmento aleatorio ---
    def process_audio(self, speech):
        len_speech = len(speech)
        maximum_ini = np.maximum(len_speech - self.seg_samples, 1)
        seg_ini = np.random.randint(maximum_ini)
        seg_end = seg_ini + self.seg_samples
        seg_speech = speech[seg_ini:seg_end]
        return seg_speech

    def trim_speech(self, speech):
        trim_speech = librosa.effects.trim(speech, top_db=self.top_db)[0]
        return trim_speech